In [1]:
# Parameters
input_file = "/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/L-alanine_neutron_295K_278464_unopt_pbesol_SEDC_magres.magres"


This code uses papermill on terminal to change input_file. The input_file is the file path to the magres file

*Make sure you change the nucleus information like Q value and nucleus (I am not sure how to do this in an efficient way)*

Shiva Agarwal

*Apr 23 2025*

In [2]:
import os
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms

atoms = MagresAtoms.load_magres(input_file)


In [3]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [4]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [5]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [6]:
nucleus = 'O'      # nucleus for which parameters are wanted
atom_label = 0      # site for which parameters wanted
Q = -0.0256         #electric quadrupole moment for nucleus in barn

In [7]:
for atom in atoms.species(nucleus):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

17O1 sigma:
 [[-63.8705543  165.38246268 -76.39960459]
 [168.45997403  77.7836575   26.01812349]
 [  7.27554215 -23.27718427 -49.85309592]]

17O2 sigma:
 [[ -63.8705543  -165.38246268   76.39960459]
 [-168.45997403   77.7836575    26.01812349]
 [  -7.27554215  -23.27718427  -49.85309592]]

17O3 sigma:
 [[-63.8705543  165.38246268  76.39960459]
 [168.45997403  77.7836575  -26.01812349]
 [ -7.27554215  23.27718427 -49.85309592]]

17O4 sigma:
 [[ -63.8705543  -165.38246268  -76.39960459]
 [-168.45997403   77.7836575   -26.01812349]
 [   7.27554215   23.27718427  -49.85309592]]

17O5 sigma:
 [[ -51.13735337  209.78151665   55.79424213]
 [ 180.6729838   120.84093606  -83.96303921]
 [  19.52691974  -46.88498251 -161.70121811]]

17O6 sigma:
 [[ -51.13735337 -209.78151665  -55.79424213]
 [-180.6729838   120.84093606  -83.96303921]
 [ -19.52691974  -46.88498251 -161.70121811]]

17O7 sigma:
 [[ -51.13735337  209.78151665  -55.79424213]
 [ 180.6729838   120.84093606   83.96303921]
 [ -19.52691974

In [8]:
for atom in atoms.species('O'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

17O1 sigma:
 6.487180789800043

17O2 sigma:
 6.487180789800044

17O3 sigma:
 6.487180789800051

17O4 sigma:
 6.487180789800073

17O5 sigma:
 8.080015858751885

17O6 sigma:
 8.0800158587519

17O7 sigma:
 8.080015858751935

17O8 sigma:
 8.080015858751947



In [9]:
# using values from latest magres file
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + 2) Tensor from updated_magres
CS_anti = np.zeros((3,3))                           # CS antisymmetric ( l = 1)
CS_iso = np.zeros((3,3))                            # CS isotropic  (l = 0)
CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + 1 + 2) Tensor from magres

                                         
CS_total[:,:] = atoms.species(nucleus).ms.sigma[atom_label]

iso = np.mean([CS_total[0,0], CS_total[1,1], CS_total[2,2]]) # isotropic chemical shielding (l = 0)

CS_iso[0,0] = CS_iso[1,1] = CS_iso[2,2] = iso

Cs[0,0] = CS_total[0,0]; Cs[0,1] = (CS_total[0,1] + CS_total[1,0] )/2; Cs[0,2] = (CS_total[0,2] + CS_total[2,0])/2;
Cs[1,0] = Cs[0,1];      Cs[1,1] = CS_total[1,1];                      Cs[1,2] = (CS_total[1,2] + CS_total[2,1])/2;
Cs[2,0] = Cs[0,2];      Cs[2,1] = Cs[1,2];                           Cs[2,2] = CS_total[2,2];

CS_anti[0,1] = (CS_total[0,1] - CS_total[1,0])/2; CS_anti[0,2] = (CS_total[0,2] - CS_total[2,0])/2; 
CS_anti[1,0] = -CS_anti[0,1]; CS_anti[1,2] = (CS_total[1,2] - CS_total[2,1])/2;
CS_anti[2,0] = -CS_anti[0,2];      CS_anti[2,1] = -CS_anti[1,2]; 


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres (in a.u.)

efg[:,:] = atoms.species(nucleus)[atom_label].efg.V


# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

V = efg*Q*234.9647

print('\nQ tensor:\n', np.round(V,3))
print('\nCS Tensor:\n',np.round(CS_total, 3))
print('\nCS isotropic Tensor:\n',np.round(CS_iso, 3))
print('\nCS symmetric Tensor:\n',np.round(Cs,3))
print('\nCS antisymmetric Tensor:\n',np.round(CS_anti,3))



Q tensor:
 [[ 0.942 -1.012  4.634]
 [-1.012 -0.817 -3.58 ]
 [ 4.634 -3.58  -0.125]]

CS Tensor:
 [[-63.871 165.382 -76.4  ]
 [168.46   77.784  26.018]
 [  7.276 -23.277 -49.853]]

CS isotropic Tensor:
 [[-11.98   0.     0.  ]
 [  0.   -11.98   0.  ]
 [  0.     0.   -11.98]]

CS symmetric Tensor:
 [[-63.871 166.921 -34.562]
 [166.921  77.784   1.37 ]
 [-34.562   1.37  -49.853]]

CS antisymmetric Tensor:
 [[  0.     -1.539 -41.838]
 [  1.539   0.     24.648]
 [ 41.838 -24.648   0.   ]]


In [10]:
print("For EFG tensor")
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
print("For CS tensor")
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

For EFG tensor
 Unsorted Eigenvalues:
 [ 6.49225064 -1.1279393  -5.36431134] 

 Unsorted Eigenvectors:
 [[-0.62731513  0.62236558 -0.46812051]
 [ 0.41087307  0.77513685  0.47994395]
 [-0.66155805 -0.10873799  0.74196832]] 

Sorted Eigenvalues: 
 [-1.1279393  -5.36431134  6.49225064] 

Sorted Eigenvectors: 
 [[ 0.62236558 -0.46812051 -0.62731513]
 [ 0.77513685  0.47994395  0.41087307]
 [-0.10873799  0.74196832 -0.66155805]] 


For CS tensor
 Unsorted Eigenvalues:
 [ 189.63945943 -181.07998057  -44.49947158] 

 Unsorted Eigenvectors:
 [[ 0.55552108  0.81906409 -0.14328414]
 [ 0.82807399 -0.52932267  0.18469156]
 [-0.07543068  0.22124993  0.97229557]] 

Sorted Eigenvalues: 
 [ -44.49947158 -181.07998057  189.63945943] 

Sorted Eigenvectors: 
 [[-0.14328414  0.81906409  0.55552108]
 [ 0.18469156 -0.52932267  0.82807399]
 [ 0.97229557  0.22124993 -0.07543068]] 



In [11]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -1.12793929944519 -5.364311338550734 6.492250637995981
CSA Tensor Components δyy, δxx, δzz: 
 -44.49947158221846 -181.079980567897 189.6394594298309


In [12]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]

table_string = tabulate(table, headers=['Quantity', 'Value'], tablefmt='grid')
print(f'Parameters for {atoms.species(nucleus)[atom_label]}: \n', table_string)

Parameters for 17O1: 
 +--------------+------------+
| Quantity     |      Value |
+==============+============+
| CQ (MHz)     |   6.49225  |
+--------------+------------+
| etaq         |   0.652527 |
+--------------+------------+
| iso_cs (ppm) | -11.98     |
+--------------+------------+
| csa (ppm)    | 201.619    |
+--------------+------------+
| etas         |   0.677417 |
+--------------+------------+


In [13]:

# Derive output name from input file
base_name = os.path.splitext(os.path.basename(input_file))[0]
output_txt = f"/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/{nucleus}_all_results.txt"

# Save to .txt file
with open(output_txt, 'a') as f:
    f.write(f"\n\n===== Results for: {base_name} =====\n\n")
    f.write(table_string)
    f.write("\n")

print(f"Saved table to {output_txt}")

Saved table to /home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/NMR_analysis/output_txt/O_all_results.txt


In [14]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[-0.46812051  0.62236558 -0.62731513]
 [ 0.47994395  0.77513685  0.41087307]
 [ 0.74196832 -0.10873799 -0.66155805]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-8.337538984215646 131.4188071703964 33.223626644961186 

Direction cosine csa: 

[[ 0.81906409 -0.14328414  0.55552108]
 [-0.52932267  0.18469156  0.82807399]
 [ 0.22124993  0.97229557 -0.07543068]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
77.18040032454877 94.32596866225467 -56.14395052626293 



In [15]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: 28.546739296705518 chi: 87.61303783945924 xi: -85.32625463295666 

